In [0]:
%pip install fpdf2 requests -q

In [0]:
import os
import json
import requests
import time
from fpdf import FPDF

# ==========================================
# 1. API Keys Configuration
# ==========================================
NVIDIA_NIM_API_KEY = dbutils.secrets.get("jobs_automation", "nvidia_nemotron_3_ultra")
SAVE_DIRECTORY = "/Workspace/Users/maheshmahi1282@gmail.com/Jobs_Automation/Drafts/"
CACHE_FILE = os.path.join(SAVE_DIRECTORY, "cached_jobs.json")

# ==========================================
# 2. Base Profile Information
# ==========================================
BASE_RESUME_INFO = {
    "name": "Sumanth Madduluri",
    "contact": "Ongole, AP, India | Phone: +91-XXXXXXXXXX | Email: sumanth@example.com",
    "links": "LinkedIn: linkedin.com/in/sumanth | GitHub: github.com/sumanth",
    "education": [
        "Master of Science in Computer Science (M.Sc) - 2025",
        "Bachelor of Commerce in Computers (B.Com) - 2023"
    ]
}

# ==========================================
# 3. Human-Like AI Engine
# ==========================================
def generate_ats_content(company_name, job_title, job_description):
    print(f"🧠 NVIDIA AI is engineering a human-like resume for {company_name}...")
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    You are an expert, highly experienced human ATS Resume Writer. Your task is to generate a comprehensive, natural-sounding resume tailored for the {job_title} role at {company_name}.
    
    Job Description: 
    {job_description}
    
    My Base Profile:
    - Base Skills: Python (3.12), PySpark, Databricks, SQL, AWS S3, Cloudflare, Linux, Git, Delta Lake.
    - Experience: Software Engineer at SPINO Technologies (May 2026 - Present).
    - Project 1: Snap On Wheels (Python Flask, Digital Imaging, Automation).
    - Project 2: Courseify (Mobile backend, GitHub, Cloud caching).

    CRITICAL INSTRUCTIONS FOR HUMAN-LIKE WRITING:
    1. AVOID ROBOTIC REPETITION. Do not start every bullet point with the same word. Vary your sentence structures naturally as a real professional human would.
    2. Seamlessly and naturally blend the core requirements of {company_name} into my SPINO Technologies experience and my Projects. 
    3. Make it sound authentic, impactful, and conversational yet highly professional.
    4. Write 6 to 8 engaging bullet points for SPINO Technologies.
    5. Write 4 engaging bullet points for EACH project.
    6. Include a "core_competencies" array with 6-8 ATS keywords directly from the JD.
    
    Return ONLY a valid JSON object matching exactly this structure:
    {{
        "summary": "A highly detailed, compelling and natural-sounding 4-6 sentence summary.",
        "core_competencies": ["Keyword 1", "Keyword 2", ...],
        "skills": {{
            "Languages": ["..."],
            "Big Data & Cloud": ["..."],
            "Frameworks & Tools": ["..."]
        }},
        "experience_bullets": [
            "Human-sounding detailed bullet 1",
            "Human-sounding detailed bullet 2",
            "... up to 8 bullets"
        ],
        "project1_bullets": ["Bullet 1", "Bullet 2", "Bullet 3", "Bullet 4"],
        "project2_bullets": ["Bullet 1", "Bullet 2", "Bullet 3", "Bullet 4"]
    }}
    """
    
    payload = {
        "model": "meta/llama-3.1-70b-instruct", 
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.3, # Slightly increased for more natural, creative phrasing
        "max_tokens": 2500
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            content = response.json()['choices'][0]['message']['content'].strip()
            start_idx = content.find('{')
            end_idx = content.rfind('}')
            if start_idx != -1 and end_idx != -1:
                clean_json = content[start_idx:end_idx+1]
                return json.loads(clean_json)
            else:
                print("❌ Failed to find JSON block in AI response.")
                return None
        else:
            print(f"❌ NVIDIA API Error: {response.text}")
            return None
    except Exception as e:
        print(f"❌ JSON Parse Error: {str(e)}")
        return None

# ==========================================
# 4. ATS PDF Engine (With Pagination Fix)
# ==========================================
class ATSResumePDF(FPDF):
    def check_space(self, required_space):
        # If the remaining space on the page is less than required, add a new page
        # A4 page height is approx 297mm. Bottom margin is usually 15mm.
        if self.get_y() + required_space > 280:
            self.add_page()

    def add_section_header(self, title):
        # Guarantee at least 35mm of space before printing a header so content doesn't break
        self.check_space(35)
        self.ln(6)
        self.set_font("Arial", "B", 12)
        self.set_text_color(44, 62, 80)
        self.cell(0, 6, title.upper(), ln=True)
        self.set_draw_color(189, 195, 199)
        self.set_line_width(0.5)
        self.line(self.get_x(), self.get_y(), self.get_x() + 190, self.get_y())
        self.ln(4)

def render_bullet_list(pdf, bullets_data):
    if isinstance(bullets_data, str):
        bullets_data = [bullets_data]
    elif not isinstance(bullets_data, list):
        bullets_data = []

    pdf.set_font("Arial", "", 10.5)
    pdf.set_text_color(40, 40, 40)
    for bullet in bullets_data:
        # Check space for each bullet point to prevent awkward cuts
        pdf.check_space(10)
        pdf.set_x(15) 
        pdf.multi_cell(0, 6, f"{chr(149)}  {bullet.strip()}")
        pdf.ln(1.5)

def create_resume_pdf(ai_data, company, role, filename):
    pdf = ATSResumePDF(orientation='P', unit='mm', format='A4')
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    
    # --- HEADER ---
    pdf.set_font("Arial", "B", 22)
    pdf.set_text_color(44, 62, 80)
    pdf.cell(0, 9, BASE_RESUME_INFO["name"], ln=True, align='C')
    
    pdf.set_font("Arial", "", 10)
    pdf.set_text_color(100, 100, 100)
    pdf.cell(0, 5, BASE_RESUME_INFO["contact"], ln=True, align='C')
    pdf.cell(0, 5, BASE_RESUME_INFO["links"], ln=True, align='C')
    
    pdf.ln(3)
    pdf.set_font("Arial", "I", 10)
    pdf.set_text_color(52, 152, 219)
    pdf.cell(0, 5, f"Tailored exclusively for: {role} @ {company}", ln=True, align='C')
    
    # --- SUMMARY ---
    pdf.add_section_header("Professional Summary")
    pdf.set_font("Arial", "", 10.5)
    pdf.set_text_color(40, 40, 40)
    pdf.multi_cell(0, 6, ai_data.get("summary", ""))
    
    # --- CORE COMPETENCIES ---
    if ai_data.get("core_competencies"):
        pdf.add_section_header("Core Competencies")
        pdf.set_font("Arial", "B", 10.5)
        comps_str = "  |  ".join(ai_data.get("core_competencies", []))
        pdf.multi_cell(0, 6, comps_str, align='C')

    # --- TECHNICAL SKILLS ---
    pdf.add_section_header("Technical Skills")
    skills = ai_data.get("skills", {})
    if isinstance(skills, dict):
        for category, items in skills.items():
            pdf.check_space(8)
            pdf.set_font("Arial", "B", 10.5)
            pdf.write(7, f"{category}: ")
            pdf.set_font("Arial", "", 10.5)
            items_str = ", ".join(items) if isinstance(items, list) else str(items)
            pdf.write(7, f"{items_str}\n")

    # --- PROFESSIONAL EXPERIENCE ---
    pdf.add_section_header("Professional Experience")
    pdf.set_font("Arial", "B", 11.5)
    pdf.cell(130, 7, "Software Engineer | SPINO Technologies")
    pdf.set_font("Arial", "I", 10.5)
    pdf.cell(60, 7, "May 2026 - Present", ln=True, align='R')
    pdf.ln(2)
    render_bullet_list(pdf, ai_data.get("experience_bullets", []))

    # --- KEY PROJECTS ---
    pdf.add_section_header("Key Projects")
    
    pdf.check_space(15)
    pdf.set_font("Arial", "B", 11.5)
    pdf.cell(0, 7, "Snap On Wheels | Backend Architecture & Automation", ln=True)
    pdf.ln(1)
    render_bullet_list(pdf, ai_data.get("project1_bullets", []))
    pdf.ln(3)
    
    pdf.check_space(15)
    pdf.set_font("Arial", "B", 11.5)
    pdf.cell(0, 7, "Courseify | Cloud Infrastructure & Caching Strategy", ln=True)
    pdf.ln(1)
    render_bullet_list(pdf, ai_data.get("project2_bullets", []))

    # --- EDUCATION ---
    pdf.add_section_header("Education")
    render_bullet_list(pdf, BASE_RESUME_INFO["education"])

    pdf.output(filename)
    print(f"📄 Human-Like Resume Ready: {filename}")

# ==========================================
# 5. OFFLINE TESTING PIPELINE
# ==========================================
def run_test_pipeline():
    print("🛠️ OFFLINE MODE: Reading locally cached jobs to save API limit...")
    try:
        with open(CACHE_FILE, "r") as f:
            jobs_list = json.load(f)
    except FileNotFoundError:
        print(f"🛑 Error: '{CACHE_FILE}' not found. Run the fetch logic once.")
        return
        
    print(f"✅ Loaded {len(jobs_list)} jobs from cache. Generating Human-Like PDFs...\n")
    
    for idx, job in enumerate(jobs_list[:3]):
        company_name = job.get('employer_name', f'Company_{idx}')
        job_title = job.get('job_title', 'Data Engineer')
        job_desc = job.get('job_description', '')
        
        print(f"🚀 Testing Job {idx+1}: {job_title} at {company_name}")
        
        ai_data = generate_ats_content(company_name, job_title, job_desc)
        
        if ai_data:
            clean_company = re.sub(r'[^a-zA-Z0-9]', '_', company_name)
            output_file = os.path.join(SAVE_DIRECTORY, f"Sumanth_Perfected_{clean_company}.pdf")
            create_resume_pdf(ai_data, company_name, job_title, output_file)
            
        time.sleep(2) 

run_test_pipeline()

In [0]:
import os
import json
import requests
import time
from fpdf import FPDF

# ==========================================
# 1. API Keys Configuration
# ==========================================
NVIDIA_NIM_API_KEY = dbutils.secrets.get("jobs_automation", "nvidia_nemotron_3_ultra")
SAVE_DIRECTORY = "/Workspace/Users/maheshmahi1282@gmail.com/Jobs_Automation/Drafts/"
CACHE_FILE = os.path.join(SAVE_DIRECTORY, "cached_jobs.json")

# ==========================================
# 2. Base Profile Information
# ==========================================
BASE_RESUME_INFO = {
    "name": "Sumanth Madduluri",
    "contact": "Ongole, AP, India | Phone: +91-XXXXXXXXXX | Email: sumanth@example.com",
    "links": "LinkedIn: linkedin.com/in/sumanth | GitHub: github.com/sumanth",
    "education": [
        "Master of Science in Computer Science (M.Sc) - 2025",
        "Bachelor of Commerce in Computers (B.Com) - 2023"
    ]
}

# ==========================================
# 3. Intelligent AI Engine (JD Injection + 2 Page Expander)
# ==========================================
def generate_ats_content(company_name, job_title, job_description):
    print(f"🧠 NVIDIA AI is analyzing JD and engineering a 2-page resume for {company_name}...")
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    You are an elite ATS Resume Writer. Your task is to generate a comprehensive, multi-page resume tailored for the {job_title} role at {company_name}.
    
    Job Description: 
    {job_description}
    
    My Base Profile:
    - Base Skills: Python (3.12), PySpark, Databricks, SQL, AWS S3, Cloudflare, Linux, Git, Delta Lake.
    - Experience: Software Engineer at SPINO Technologies (May 2026 - Present).
    - Project 1: Snap On Wheels (Python Flask, Digital Imaging, Automation).
    - Project 2: Courseify (Mobile backend, GitHub, Cloud caching).

    CRITICAL INSTRUCTIONS FOR JD INJECTION & LENGTH:
    1. Analyze the JD and identify {company_name}'s core focus areas, target projects, and specific tech stack.
    2. Seamlessly inject these focus areas into my SPINO Technologies experience and my Projects. Make it look like I have extensively worked on the exact things they are looking for.
    3. Generate highly detailed, extensive content to ensure the final resume spans at least 2 pages.
    4. Write 8 to 10 long, metric-driven bullet points (XYZ format) for SPINO Technologies.
    5. Write 5 long bullet points for EACH project.
    6. Include a "core_competencies" array with 8-10 ATS keywords directly from the JD.
    
    Return ONLY a valid JSON object matching exactly this structure:
    {{
        "summary": "A highly detailed, compelling 6-sentence summary.",
        "core_competencies": ["Keyword 1", "Keyword 2", ...],
        "skills": {{
            "Languages": ["..."],
            "Big Data & Cloud": ["..."],
            "Frameworks & Tools": ["..."]
        }},
        "experience_bullets": [
            "Bullet 1 (Long, detailed, JD injected)",
            "Bullet 2...",
            "... up to 10 bullets"
        ],
        "project1_bullets": ["Bullet 1", "Bullet 2", "Bullet 3", "Bullet 4", "Bullet 5"],
        "project2_bullets": ["Bullet 1", "Bullet 2", "Bullet 3", "Bullet 4", "Bullet 5"]
    }}
    """
    
    payload = {
        "model": "meta/llama-3.1-70b-instruct", 
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2, 
        "max_tokens": 2500 # Increased for longer content
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            content = response.json()['choices'][0]['message']['content'].strip()
            # BULLETPROOF JSON PARSER: Finds the first { and last }
            start_idx = content.find('{')
            end_idx = content.rfind('}')
            if start_idx != -1 and end_idx != -1:
                clean_json = content[start_idx:end_idx+1]
                return json.loads(clean_json)
            else:
                print("❌ Failed to find JSON block in AI response.")
                print("Raw output:", content)
                return None
        else:
            print(f"❌ NVIDIA API Error: {response.text}")
            return None
    except Exception as e:
        print(f"❌ JSON Parse Error: {str(e)}")
        return None

# ==========================================
# 4. ATS PDF Engine (Multi-Page Supported)
# ==========================================
class ATSResumePDF(FPDF):
    def add_section_header(self, title):
        self.ln(6)
        self.set_font("Arial", "B", 12)
        self.set_text_color(44, 62, 80)
        self.cell(0, 6, title.upper(), ln=True)
        self.set_draw_color(189, 195, 199)
        self.set_line_width(0.5)
        self.line(self.get_x(), self.get_y(), self.get_x() + 190, self.get_y())
        self.ln(4)

def render_bullet_list(pdf, bullets_data):
    if isinstance(bullets_data, str):
        bullets_data = [bullets_data]
    elif not isinstance(bullets_data, list):
        bullets_data = []

    pdf.set_font("Arial", "", 10.5)
    pdf.set_text_color(40, 40, 40)
    for bullet in bullets_data:
        pdf.set_x(15) 
        # Multi-cell auto-wraps text and handles page breaks
        pdf.multi_cell(0, 6, f"{chr(149)}  {bullet.strip()}")
        pdf.ln(1.5)

def create_resume_pdf(ai_data, company, role, filename):
    pdf = ATSResumePDF(orientation='P', unit='mm', format='A4')
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    
    # --- HEADER ---
    pdf.set_font("Arial", "B", 22)
    pdf.set_text_color(44, 62, 80)
    pdf.cell(0, 9, BASE_RESUME_INFO["name"], ln=True, align='C')
    
    pdf.set_font("Arial", "", 10)
    pdf.set_text_color(100, 100, 100)
    pdf.cell(0, 5, BASE_RESUME_INFO["contact"], ln=True, align='C')
    pdf.cell(0, 5, BASE_RESUME_INFO["links"], ln=True, align='C')
    
    pdf.ln(3)
    pdf.set_font("Arial", "I", 10)
    pdf.set_text_color(52, 152, 219)
    pdf.cell(0, 5, f"Tailored exclusively for: {role} @ {company}", ln=True, align='C')
    
    # --- SUMMARY ---
    pdf.add_section_header("Professional Summary")
    pdf.set_font("Arial", "", 10.5)
    pdf.set_text_color(40, 40, 40)
    pdf.multi_cell(0, 6, ai_data.get("summary", ""))
    
    # --- CORE COMPETENCIES ---
    pdf.add_section_header("Core Competencies")
    pdf.set_font("Arial", "B", 10.5)
    competencies = ai_data.get("core_competencies", [])
    if competencies:
        comps_str = "  |  ".join(competencies)
        pdf.multi_cell(0, 6, comps_str, align='C')

    # --- TECHNICAL SKILLS ---
    pdf.add_section_header("Technical Skills")
    skills = ai_data.get("skills", {})
    if isinstance(skills, dict):
        for category, items in skills.items():
            pdf.set_font("Arial", "B", 10.5)
            pdf.write(7, f"{category}: ")
            pdf.set_font("Arial", "", 10.5)
            items_str = ", ".join(items) if isinstance(items, list) else str(items)
            pdf.write(7, f"{items_str}\n")

    # --- PROFESSIONAL EXPERIENCE ---
    pdf.add_section_header("Professional Experience")
    pdf.set_font("Arial", "B", 11.5)
    pdf.cell(130, 7, "Software Engineer | SPINO Technologies")
    pdf.set_font("Arial", "I", 10.5)
    pdf.cell(60, 7, "May 2026 - Present", ln=True, align='R')
    pdf.ln(2)
    render_bullet_list(pdf, ai_data.get("experience_bullets", []))

    # --- KEY PROJECTS ---
    pdf.add_section_header("Key Projects")
    
    pdf.set_font("Arial", "B", 11.5)
    pdf.cell(0, 7, "Snap On Wheels | Backend Architecture & Automation", ln=True)
    pdf.ln(1)
    render_bullet_list(pdf, ai_data.get("project1_bullets", []))
    pdf.ln(3)
    
    pdf.set_font("Arial", "B", 11.5)
    pdf.cell(0, 7, "Courseify | Cloud Infrastructure & Caching Strategy", ln=True)
    pdf.ln(1)
    render_bullet_list(pdf, ai_data.get("project2_bullets", []))

    # --- EDUCATION ---
    pdf.add_section_header("Education")
    render_bullet_list(pdf, BASE_RESUME_INFO["education"])

    pdf.output(filename)
    print(f"📄 Detailed 2+ Page Resume Ready: {filename}")

# ==========================================
# 5. OFFLINE TESTING PIPELINE
# ==========================================
def run_test_pipeline():
    print("🛠️ OFFLINE MODE: Reading locally cached jobs to save API limit...")
    try:
        with open(CACHE_FILE, "r") as f:
            jobs_list = json.load(f)
    except FileNotFoundError:
        print(f"🛑 Error: '{CACHE_FILE}' not found. Run the fetch logic once.")
        return
        
    print(f"✅ Loaded {len(jobs_list)} jobs from cache. Generating heavy content PDFs...\n")
    
    for idx, job in enumerate(jobs_list[:3]):
        company_name = job.get('employer_name', f'Company_{idx}')
        job_title = job.get('job_title', 'Data Engineer')
        job_desc = job.get('job_description', '')
        
        print(f"🚀 Testing Job {idx+1}: {job_title} at {company_name}")
        
        ai_data = generate_ats_content(company_name, job_title, job_desc)
        
        if ai_data:
            clean_company = re.sub(r'[^a-zA-Z0-9]', '_', company_name)
            output_file = os.path.join(SAVE_DIRECTORY, f"Sumanth_Heavy_Resume_{clean_company}.pdf")
            create_resume_pdf(ai_data, company_name, job_title, output_file)
            
        time.sleep(2) 

run_test_pipeline()

In [0]:
import os
import json
import re
import requests
import time
from fpdf import FPDF

# ==========================================
# 1. API Keys (Only NVIDIA Needed Now)
# ==========================================
NVIDIA_NIM_API_KEY = dbutils.secrets.get("jobs_automation", "nvidia_nemotron_3_ultra")
SAVE_DIRECTORY = "/Workspace/Users/maheshmahi1282@gmail.com/Jobs_Automation/Drafts/"
CACHE_FILE = os.path.join(SAVE_DIRECTORY, "cached_jobs.json")

# ==========================================
# 2. Detailed Profile Base
# ==========================================
BASE_RESUME_INFO = {
    "name": "Sumanth Madduluri",
    "contact": "Ongole, AP, India | Phone: +91-XXXXXXXXXX | Email: sumanth@example.com",
    "links": "LinkedIn: linkedin.com/in/sumanth | GitHub: github.com/sumanth",
    "education": [
        "Master of Science in Computer Science (M.Sc) - 2025",
        "Bachelor of Commerce in Computers (B.Com) - 2023"
    ]
}

# ==========================================
# 3. Strictly Templated AI Engine (Llama 3.1)
# ==========================================
def generate_ats_content(company_name, job_title, job_description):
    print(f"🧠 NVIDIA AI is engineering resume for {company_name}...")
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    You are an expert ATS Resume Writer. Tailor my resume for the {job_title} role at {company_name}.
    
    Job Description: {job_description}
    
    My Base Profile:
    - Skills: Python (3.12), PySpark, Databricks, SQL, AWS S3, Cloudflare, Linux, Git, Delta Lake.
    - Experience: Software Engineer at SPINO Technologies (May 2026 - Present). Built data pipelines, optimized Apache Spark/Delta Lake tables, automated AWS S3/Cloudflare workflows.
    - Project 1: Snap On Wheels. Automated photo booth software built with Python Flask and digital imaging pipelines.
    - Project 2: Courseify. Mobile learning app using GitHub and cloud caching.
    
    You MUST return ONLY a valid JSON object matching exactly this structure (Ensure arrays are used for bullets):
    {{
        "summary": "Compelling 3-sentence summary targeting this specific job.",
        "skills": {{
            "Languages": ["Python", "SQL"],
            "Big Data & Cloud": ["PySpark", "Databricks", "AWS S3"],
            "Tools": ["Git", "Linux", "Cloudflare"]
        }},
        "experience_bullets": [
            "Accomplished X by doing Y resulting in Z.",
            "Bullet 2 with metrics.",
            "Bullet 3 showing impact.",
            "Bullet 4 focused on tech."
        ],
        "project1_bullets": [
            "Impactful bullet 1 for Snap On Wheels.",
            "Impactful bullet 2 for Snap On Wheels."
        ],
        "project2_bullets": [
            "Impactful bullet 1 for Courseify.",
            "Impactful bullet 2 for Courseify."
        ]
    }}
    """
    
    payload = {
        "model": "meta/llama-3.1-70b-instruct", 
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1, # Lowered temperature for stricter JSON adherence
        "max_tokens": 1500
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            content = response.json()['choices'][0]['message']['content'].strip()
            clean_json = re.sub(r"^```json\s*|```$", "", content, flags=re.MULTILINE).strip()
            return json.loads(clean_json)
        else:
            print(f"❌ NVIDIA API Error: {response.text}")
            return None
    except Exception as e:
        print(f"❌ JSON Parse Error: {str(e)}")
        return None

# ==========================================
# 4. Bulletproof ATS PDF Engine
# ==========================================
class ATSResumePDF(FPDF):
    def add_section_header(self, title):
        self.ln(4)
        self.set_font("Arial", "B", 11)
        self.set_text_color(44, 62, 80)
        self.cell(0, 6, title.upper(), ln=True)
        self.set_draw_color(189, 195, 199)
        self.set_line_width(0.5)
        self.line(self.get_x(), self.get_y(), self.get_x() + 190, self.get_y())
        self.ln(3)

def render_bullet_list(pdf, bullets_data):
    # Safety check: If AI returns a string instead of a list, convert it to a list
    if isinstance(bullets_data, str):
        bullets_data = [bullets_data]
    elif not isinstance(bullets_data, list):
        bullets_data = []

    pdf.set_font("Arial", "", 10)
    pdf.set_text_color(53, 53, 53)
    for bullet in bullets_data:
        # FPDF bullet point character is chr(149)
        pdf.set_x(15) # Indentation
        pdf.multi_cell(0, 5.5, f"{chr(149)} {bullet.strip()}")

def create_resume_pdf(ai_data, company, role, filename):
    pdf = ATSResumePDF(orientation='P', unit='mm', format='A4')
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    
    # --- HEADER ---
    pdf.set_font("Arial", "B", 20)
    pdf.set_text_color(44, 62, 80)
    pdf.cell(0, 8, BASE_RESUME_INFO["name"], ln=True, align='C')
    
    pdf.set_font("Arial", "", 9.5)
    pdf.set_text_color(127, 140, 141)
    pdf.cell(0, 5, BASE_RESUME_INFO["contact"], ln=True, align='C')
    pdf.cell(0, 5, BASE_RESUME_INFO["links"], ln=True, align='C')
    
    pdf.ln(2)
    pdf.set_font("Arial", "I", 9)
    pdf.set_text_color(52, 152, 219)
    pdf.cell(0, 5, f"Tailored exclusively for: {role} @ {company}", ln=True, align='C')
    
    # --- SUMMARY ---
    pdf.add_section_header("Professional Summary")
    pdf.set_font("Arial", "", 10)
    pdf.set_text_color(53, 53, 53)
    pdf.multi_cell(0, 5.5, ai_data.get("summary", ""))
    
    # --- SKILLS ---
    pdf.add_section_header("Technical Skills")
    skills = ai_data.get("skills", {})
    if isinstance(skills, dict):
        for category, items in skills.items():
            pdf.set_font("Arial", "B", 10)
            pdf.write(6, f"{category}: ")
            pdf.set_font("Arial", "", 10)
            items_str = ", ".join(items) if isinstance(items, list) else str(items)
            pdf.write(6, f"{items_str}\n")

    # --- EXPERIENCE ---
    pdf.add_section_header("Professional Experience")
    pdf.set_font("Arial", "B", 10.5)
    pdf.cell(130, 6, "Software Engineer | SPINO Technologies")
    pdf.set_font("Arial", "I", 9.5)
    pdf.cell(60, 6, "May 2026 - Present", ln=True, align='R')
    render_bullet_list(pdf, ai_data.get("experience_bullets", []))

    # --- PROJECTS ---
    pdf.add_section_header("Key Projects")
    
    pdf.set_font("Arial", "B", 10.5)
    pdf.cell(0, 6, "Snap On Wheels | Python, Flask, Digital Imaging Pipeline", ln=True)
    render_bullet_list(pdf, ai_data.get("project1_bullets", []))
    pdf.ln(2)
    
    pdf.set_font("Arial", "B", 10.5)
    pdf.cell(0, 6, "Courseify | GitHub, Cloud Caching, Mobile App Backend", ln=True)
    render_bullet_list(pdf, ai_data.get("project2_bullets", []))

    # --- EDUCATION ---
    pdf.add_section_header("Education")
    render_bullet_list(pdf, BASE_RESUME_INFO["education"])

    pdf.output(filename)
    print(f"📄 High-Impact Resume Ready: {filename}")

# ==========================================
# 5. OFFLINE TESTING PIPELINE (No JSearch API)
# ==========================================
def run_test_pipeline():
    print("🛠️ OFFLINE MODE: Reading locally cached jobs to save API limit...")
    try:
        with open(CACHE_FILE, "r") as f:
            jobs_list = json.load(f)
    except FileNotFoundError:
        print(f"🛑 Error: '{CACHE_FILE}' not found. You need to run JSearch at least once.")
        return
        
    print(f"✅ Loaded {len(jobs_list)} jobs from cache. Generating test PDFs for top 3...\n")
    
    for idx, job in enumerate(jobs_list[:3]):
        company_name = job.get('employer_name', f'Company_{idx}')
        job_title = job.get('job_title', 'Data Engineer')
        job_desc = job.get('job_description', '')
        
        print(f"🚀 Testing Job {idx+1}: {job_title} at {company_name}")
        
        ai_data = generate_ats_content(company_name, job_title, job_desc)
        
        if ai_data:
            clean_company = re.sub(r'[^a-zA-Z0-9]', '_', company_name)
            output_file = os.path.join(SAVE_DIRECTORY, f"Sumanth_TEST_{clean_company}.pdf")
            create_resume_pdf(ai_data, company_name, job_title, output_file)
            
        time.sleep(2) 

run_test_pipeline()

In [0]:
import os
import json
import re
import requests
from fpdf import FPDF

# ==========================================
# 1. API Configurations
# ==========================================
JSEARCH_API_KEY = dbutils.secrets.get("jobs_automation", "jsearch")
NVIDIA_NIM_API_KEY = dbutils.secrets.get("jobs_automation", "nvidia_nemotron_3_ultra")

# ==========================================
# 2. Detailed Profile Base (Your Exact Details)
# ==========================================
BASE_RESUME_INFO = {
    "name": "Sumanth Madduluri",
    "contact": "Ongole, AP, India | Phone: +91-XXXXXXXXXX | Email: sumanth@example.com",
    "links": "LinkedIn: linkedin.com/in/sumanth | GitHub: github.com/sumanth",
    "education": [
        "Master of Science in Computer Science (M.Sc) - 2025",
        "Bachelor of Commerce in Computers (B.Com) - 2023"
    ]
}

# ==========================================
# 3. AI Core Engine (NVIDIA LLaMA 3.1 70B)
# ==========================================
def generate_ats_content(company_name, job_title, job_description):
    print(f"🧠 NVIDIA AI is engineering resume for {company_name}...")
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    You are an expert ATS Resume Writer. I need you to tailor my resume for the {job_title} role at {company_name}.
    
    Target Job Description:
    {job_description}
    
    My Base Profile:
    - Skills: Python (3.12), PySpark, Databricks, SQL, AWS S3, Cloudflare, Linux, Git, Delta Lake.
    - Experience: Software Engineer at SPINO Technologies (May 2026 - Present). Built data pipelines, optimized Apache Spark/Delta Lake tables, automated AWS S3/Cloudflare workflows.
    - Project 1: Snap On Wheels. Automated photo booth software built with Python Flask and digital imaging pipelines.
    - Project 2: Courseify. Mobile learning app using GitHub and cloud caching for serverless content delivery.
    
    Generate a highly impactful JSON response with the following keys:
    1. "summary": A compelling 3-4 sentence summary targeting this specific job.
    2. "skills": A dictionary categorizing my skills (e.g., "Languages", "Big Data & Cloud", "Frameworks & Tools") matching the JD.
    3. "experience_bullets": A list of 4 highly detailed, metric-driven bullet points for my SPINO Technologies role. Use the XYZ formula (Accomplished X by doing Y, resulting in Z) to make them look impactful.
    4. "project1_bullets": A list of 2 impactful bullet points for 'Snap On Wheels'.
    5. "project2_bullets": A list of 2 impactful bullet points for 'Courseify'.
    
    Return ONLY valid JSON. Do not include markdown tags like ```json.
    """
    
    payload = {
        "model": "meta/llama-3.1-70b-instruct", 
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2,
        "max_tokens": 1500
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            content = response.json()['choices'][0]['message']['content'].strip()
            clean_json = re.sub(r"^```json\s*|```$", "", content, flags=re.MULTILINE).strip()
            return json.loads(clean_json)
        else:
            print(f"❌ NVIDIA API Error: {response.text}")
            return None
    except Exception as e:
        print(f"❌ JSON Parse Error: {str(e)}")
        return None

# ==========================================
# 4. Professional ATS PDF Engine
# ==========================================
class ATSResumePDF(FPDF):
    def add_section_header(self, title):
        self.ln(4)
        self.set_font("Arial", "B", 11)
        self.set_text_color(44, 62, 80)
        self.cell(0, 6, title.upper(), ln=True)
        # Add a sleek underline
        self.set_draw_color(189, 195, 199)
        self.set_line_width(0.5)
        self.line(self.get_x(), self.get_y(), self.get_x() + 190, self.get_y())
        self.ln(3)

def create_resume_pdf(ai_data, company, role, filename):
    pdf = ATSResumePDF(orientation='P', unit='mm', format='A4')
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    
    # --- HEADER ---
    pdf.set_font("Arial", "B", 20)
    pdf.set_text_color(44, 62, 80)
    pdf.cell(0, 8, BASE_RESUME_INFO["name"], ln=True, align='C')
    
    pdf.set_font("Arial", "", 9.5)
    pdf.set_text_color(127, 140, 141)
    pdf.cell(0, 5, BASE_RESUME_INFO["contact"], ln=True, align='C')
    pdf.cell(0, 5, BASE_RESUME_INFO["links"], ln=True, align='C')
    
    pdf.ln(2)
    pdf.set_font("Arial", "I", 9)
    pdf.set_text_color(52, 152, 219)
    pdf.cell(0, 5, f"Tailored exclusively for: {role} @ {company}", ln=True, align='C')
    
    # --- SUMMARY ---
    pdf.add_section_header("Professional Summary")
    pdf.set_font("Arial", "", 10)
    pdf.set_text_color(53, 53, 53)
    pdf.multi_cell(0, 5.5, ai_data.get("summary", ""))
    
    # --- SKILLS ---
    pdf.add_section_header("Technical Skills")
    skills = ai_data.get("skills", {})
    for category, items in skills.items():
        pdf.set_font("Arial", "B", 10)
        pdf.write(6, f"{category}: ")
        pdf.set_font("Arial", "", 10)
        pdf.write(6, f"{', '.join(items) if isinstance(items, list) else items}\n")

    # --- EXPERIENCE ---
    pdf.add_section_header("Professional Experience")
    pdf.set_font("Arial", "B", 10.5)
    pdf.cell(130, 6, "Software Engineer | SPINO Technologies")
    pdf.set_font("Arial", "I", 9.5)
    pdf.cell(60, 6, "May 2026 - Present", ln=True, align='R')
    
    pdf.set_font("Arial", "", 10)
    for bullet in ai_data.get("experience_bullets", []):
        pdf.cell(5, 5.5, chr(149), align='R') # Bullet dot
        pdf.multi_cell(185, 5.5, f" {bullet}")

    # --- PROJECTS ---
    pdf.add_section_header("Key Projects")
    
    # Project 1
    pdf.set_font("Arial", "B", 10.5)
    pdf.cell(0, 6, "Snap On Wheels | Python, Flask, Digital Imaging Pipeline", ln=True)
    pdf.set_font("Arial", "", 10)
    for bullet in ai_data.get("project1_bullets", []):
        pdf.cell(5, 5.5, chr(149), align='R')
        pdf.multi_cell(185, 5.5, f" {bullet}")
    pdf.ln(2)
    
    # Project 2
    pdf.set_font("Arial", "B", 10.5)
    pdf.cell(0, 6, "Courseify | GitHub, Cloud Caching, Mobile App Backend", ln=True)
    pdf.set_font("Arial", "", 10)
    for bullet in ai_data.get("project2_bullets", []):
        pdf.cell(5, 5.5, chr(149), align='R')
        pdf.multi_cell(185, 5.5, f" {bullet}")

    # --- EDUCATION ---
    pdf.add_section_header("Education")
    pdf.set_font("Arial", "", 10)
    for edu in BASE_RESUME_INFO["education"]:
        pdf.cell(5, 6, chr(149), align='R')
        pdf.multi_cell(185, 6, f" {edu}")

    pdf.output(filename)
    print(f"📄 High-Impact Resume Ready: {filename}")

# ==========================================
# 5. Pipeline Execution
# ==========================================
def run_pipeline():
    SAVE_DIRECTORY = "/Workspace/Users/maheshmahi1282@gmail.com/Jobs_Automation/Drafts/"
    CACHE_FILE = os.path.join(SAVE_DIRECTORY, "cached_jobs.json")
    
    try:
        with open(CACHE_FILE, "r") as f:
            jobs_list = json.load(f)
    except FileNotFoundError:
        print("🛑 Cached jobs not found. Please run the fetch step first.")
        return
        
    print(f"✅ Found jobs. Processing top 3 for Advanced ATS formatting...\n")
    
    for idx, job in enumerate(jobs_list[:3]):
        company_name = job.get('employer_name', f'Company_{idx}')
        job_title = job.get('job_title', 'Data Engineer')
        job_desc = job.get('job_description', '')
        
        print(f"🚀 Processing Job {idx+1}: {job_title} at {company_name}")
        
        ai_data = generate_ats_content(company_name, job_title, job_desc)
        
        if ai_data:
            clean_company = re.sub(r'[^a-zA-Z0-9]', '_', company_name)
            output_file = os.path.join(SAVE_DIRECTORY, f"Sumanth_Resume_{clean_company}.pdf")
            create_resume_pdf(ai_data, company_name, job_title, output_file)
            
        time.sleep(2) 

run_pipeline()

In [0]:
import os
import json
import re
import requests
from fpdf import FPDF

# 1. Base Resume Configuration 
BASE_RESUME_INFO = {
    "name": "Sumanth Madduluri",
    "contact": "Ongole, AP, India | Phone: +91-XXXXXXXXXX | Email: sumanth@example.com",
    "links": "LinkedIn: linkedin.com/in/sumanth | GitHub: github.com/sumanth",
    "experience": [
        {
            "role": "Data Engineer / Software Engineer",
            "company": "SPINO Technologies",
            "duration": "2024 - Present",
            "bullets": [
                "Built and optimized scalable data pipelines and Delta Lake tables using PySpark and Databricks.",
                "Engineered automated workflows with Python, Cloudflare caching, and AWS S3 lifecycle management.",
                "Designed end-to-end CI/CD and automated microservices using Flask and RESTful APIs."
            ]
        }
    ],
    "education": [
        "Master of Science in Computer Science (M.Sc)",
        "Bachelor of Commerce in Computers (B.Com)"
    ]
}

# NVIDIA_NIM_API_KEY = "YOUR_NVIDIA_API_KEY"
JSEARCH_API_KEY = dbutils.secrets.get("jobs_automation", "jsearch")
NVIDIA_NIM_API_KEY = dbutils.secrets.get("jobs_automation", "nvidia_nemotron_3_ultra")

# 2. NVIDIA AI Tailoring Function
def tailor_resume_content(company_name, job_title, job_description):
    print(f"🧠 NVIDIA AI is tailoring resume for {company_name} ({job_title})...")
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    You are an expert technical ATS resume optimizer.
    Target Company: {company_name}
    Target Role: {job_title}
    
    Job Description:
    {job_description}
    
    Base Candidate Skills:
    Python, PySpark, Databricks, SQL, AWS S3, Cloudflare, Flask, Git, CI/CD, Data Pipelines.
    
    Generate a JSON response with two keys:
    1. "summary": A compelling 3-4 sentence professional summary targeted specifically for {company_name}'s {job_title} position.
    2. "skills": A categorized list of technical skills matching the job description (Languages, Big Data & Cloud, Tools & Frameworks).
    
    Return ONLY valid JSON. No markdown backticks, no explanations.
    """
    
    payload = {
        "model": "meta/llama-3.1-70b-instruct",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2,
        "max_tokens": 1024
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            content = response.json()['choices'][0]['message']['content'].strip()
            # Clean JSON formatting if enclosed in code blocks
            clean_json = re.sub(r"^```json\s*|```$", "", content, flags=re.MULTILINE).strip()
            return json.loads(clean_json)
        else:
            print(f"❌ NVIDIA API Error ({response.status_code}): {response.text}")
            return None
    except Exception as e:
        print(f"❌ AI Extraction/JSON Parse Error: {str(e)}")
        return None


# 3. ATS-Friendly PDF Generator Class
class ResumePDF(FPDF):
    def header(self):
        pass  # Custom layout per resume

    def add_section_header(self, title):
        self.ln(3)
        self.set_font("Helvetica", "B", 11)
        self.set_text_color(30, 41, 59)
        self.cell(0, 6, title.upper(), ln=True)
        self.set_draw_color(203, 213, 225)
        self.set_line_width(0.4)
        self.line(self.get_x(), self.get_y(), self.get_x() + 190, self.get_y())
        self.ln(2)

def generate_pdf(tailored_data, company_name, job_title, output_path):
    pdf = ResumePDF(orientation='P', unit='mm', format='A4')
    pdf.set_auto_page_break(auto=True, margin=12)
    pdf.add_page()
    
    # Header: Name & Contact
    pdf.set_font("Helvetica", "B", 18)
    pdf.set_text_color(15, 23, 42)
    pdf.cell(0, 8, BASE_RESUME_INFO["name"], ln=True, align='C')
    
    pdf.set_font("Helvetica", "", 9)
    pdf.set_text_color(71, 85, 105)
    pdf.cell(0, 4, BASE_RESUME_INFO["contact"], ln=True, align='C')
    pdf.cell(0, 4, BASE_RESUME_INFO["links"], ln=True, align='C')
    
    # Target Application Meta Header
    pdf.ln(2)
    pdf.set_font("Helvetica", "I", 8.5)
    pdf.set_text_color(100, 116, 139)
    pdf.cell(0, 4, f"Tailored for: {job_title} at {company_name}", ln=True, align='C')
    
    # Summary Section
    pdf.add_section_header("Professional Summary")
    pdf.set_font("Helvetica", "", 9.5)
    pdf.set_text_color(30, 41, 59)
    summary_text = tailored_data.get("summary", "Experienced Data Engineer specializing in scalable pipelines and cloud infrastructure.")
    pdf.multi_cell(0, 4.5, summary_text)
    
    # Technical Skills Section
    pdf.add_section_header("Technical Skills")
    pdf.set_font("Helvetica", "", 9.5)
    skills = tailored_data.get("skills", {})
    if isinstance(skills, dict):
        for category, items in skills.items():
            pdf.set_font("Helvetica", "B", 9.5)
            pdf.write(4.5, f"{category}: ")
            pdf.set_font("Helvetica", "", 9.5)
            pdf.write(4.5, f"{', '.join(items) if isinstance(items, list) else items}\n")
    else:
        pdf.multi_cell(0, 4.5, str(skills))
        
    # Professional Experience Section
    pdf.add_section_header("Work Experience")
    for exp in BASE_RESUME_INFO["experience"]:
        pdf.set_font("Helvetica", "B", 10)
        pdf.cell(130, 5, f"{exp['role']} | {exp['company']}")
        pdf.set_font("Helvetica", "I", 9)
        pdf.cell(60, 5, exp['duration'], ln=True, align='R')
        
        pdf.set_font("Helvetica", "", 9)
        for bullet in exp["bullets"]:
            pdf.cell(5, 4.5, chr(149), align='R')  # Bullet point
            pdf.multi_cell(185, 4.5, f" {bullet}")
        pdf.ln(2)
        
    # Education Section
    pdf.add_section_header("Education")
    pdf.set_font("Helvetica", "", 9.5)
    for edu in BASE_RESUME_INFO["education"]:
        pdf.cell(0, 4.5, f"- {edu}", ln=True)
        
    pdf.output(output_path)
    print(f"📄 Generated ATS Resume PDF: {output_path}")


# 4. Pipeline Execution
def run_pipeline():
    # Load cached jobs or mock raw response
    try:
        with open("cached_jobs.json", "r") as f:
            jobs_data = json.load(f)
    except FileNotFoundError:
        # Sample fallback matching your test run
        jobs_data = [
            {
                "employer_name": "FlexBoard",
                "job_title": "Data Operations (DevOps) Engineer",
                "job_description": "Data platforms, CI/CD, Airflow, PySpark, AWS, Databricks..."
            }
        ]
        
    # Dict vs List check
    if isinstance(jobs_data, dict):
        jobs_list = jobs_data.get('data') or jobs_data.get('jobs') or list(jobs_data.values())
    else:
        jobs_list = jobs_data
        
    print(f"✅ Found {len(jobs_list)} jobs. Processing the first 3 jobs as a test run...\n")
    
    os.makedirs("generated_resumes", exist_ok=True)
    
    for idx, job in enumerate(jobs_list[:3]):
        company_name = job.get('employer_name', f'Company_{idx}')
        job_title = job.get('job_title', 'Data Engineer')
        job_desc = job.get('job_description', '')
        
        tailored_data = tailor_resume_content(company_name, job_title, job_desc)
        
        if tailored_data:
            clean_company = re.sub(r'[^a-zA-Z0-9]', '_', company_name)
            output_file = f"generated_resumes/Resume_{clean_company}_{idx+1}.pdf"
            generate_pdf(tailored_data, company_name, job_title, output_file)
            
    print("\n🎉 All test PDFs created successfully in folder: 'generated_resumes/'")

# Run the fixed pipeline
run_pipeline()